### Efficient TSP formulations

The TSP has more efficient formulations, since the MTZ constraints provide a weak linear relaxation. In this case, we use a Branch-and-Cut scheme by adding DFJ (Dantzig-Fulkerson-Johnson) constraints, which grow exponentially in number, but are included only when necessary. In addition, by exploiting the property of symmetric distances, we use a reduced formulation in which each city has degree 2, meaning that the traveler enters and leaves each city exactly once.

$$
\begin{aligned}
\text{Minimize} \quad & \sum_{i \in N} \sum_{\substack{j \in N \\ j > i}} c_{ij} x_{ij} \\
\text{subject to} \quad
& \sum_{j \in N,\; j \ne i} x_{ij} = 2 && \forall i \in N \\
& \sum_{i, j \in S} x_{ij} \leq |S| - 1 && \forall S \subset N,\; 2 \leq |S| \leq n - 1 \\
& x_{ij} = x_{ji} && \forall i < j,\; i,j \in N \\
& x_{ij} \in \{0,1\} && \forall i,j \in N,\; i \ne j
\end{aligned}
$$


In [1]:
%pip install gurobipy
import sys
import math
import random
from itertools import combinations
import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 21.7 MB/s eta 0:00:00


In [2]:
# Cities coordinates
cities = [
    ("Pachuca (Hidalgo)", 20.1160, -98.7330),
    ("Chetumal (Quintana Roo)", 18.5031, -88.3046),
    ("Tuxtla Gutiérrez (Chiapas)", 16.7538, -93.1131),
    ("Ciudad Victoria (Tamaulipas)", 23.7369, -99.1411),
    ("Colima (Colima)", 19.2452, -103.7249),
    ("Cuernavaca (Morelos)", 18.9242, -99.2216),
    ("Culiacán (Sinaloa)", 24.7922, -107.3940),
    ("Guadalajara (Jalisco)", 20.6597, -103.3496),
    ("Hermosillo (Sonora)", 29.0729, -110.9559),
    ("La Paz (Baja California Sur)", 24.1426, -110.3128),
    ("Mérida (Yucatán)", 20.9670, -89.6237),
    ("Mexicali (Baja California)", 32.6245, -115.4523),
    ("Monterrey (Nuevo León)", 25.6760, -100.3090),
    ("Morelia (Michoacán)", 19.7059, -101.1949),
    ("Oaxaca (Oaxaca)", 17.0732, -96.7266),
    ("Puebla (Puebla)", 19.0413, -98.2062),
    ("Querétaro (Querétaro)", 20.5888, -100.3899),
    ("San Luis Potosí (San Luis Potosí)", 22.1565, -100.9855),
    ("Tepic (Nayarit)", 21.5169, -104.8811),
    ("Tlaxcala (Tlaxcala)", 19.3191, -98.2375),
    ("Iguala (Guerrero)", 18.35, -99.5333),
    ("Xalapa (Veracruz)", 19.5438, -96.9102),
    ("Zacatecas (Zacatecas)", 22.7709, -102.5833),
    ("Villahermosa (Tabasco)", 17.9895, -92.9285),
    ("Saltillo (Coahuila)", 25.4380, -100.9737),
    ("Durango (Durango)", 24.0277, -104.6532),
    ("Guanajuato (Guanajuato)", 21.0181, -101.2574),
    ("Chihuahua (Chihuahua)", 28.6353, -106.0889),
    ("Aguascalientes (Aguascalientes)", 21.8818, -102.2950),
    ("Tlaxcala (Tlaxcala)", 19.3191, -98.2375),
    ("San Francisco de Campeche (Campeche)", 19.8454, -90.5237),
    ("Ciudad de México (Ciudad de México)", 19.4326, -99.1332),
    ("Toluca (Estado de México)", 19.2826, -99.6557)
]


n = len(cities)  # Number of cities

The first function is a Gurobi callback. Each time an integer feasible solution is found, it is retrieved and the corresponding tour is evaluated using the subtour detection function. If a subtour is found whose length is smaller than the total number of cities, a DFJ constraint is added.

The second function is designed to identify the smallest cycle in the current solution.

Finally, the last function returns a distance measure that is more representative than the Euclidean distance.

In [3]:
# Callback to eliminate subtours
def subtourelim(model, where):
    if where == GRB.Callback.MIPSOL:
        vals = model.cbGetSolution(model._vars)
        # Find the shortest subtour
        tour = subtour(vals)
        if len(tour) < n:
            # Add constraint to eliminate subtour
            model.cbLazy(
                gp.quicksum(model._vars[i, j] for i, j in combinations(tour, 2))
                <= len(tour) - 1
            )

# Find the smallest subtour
def subtour(vals):
    # List of selected edges in the solution
    edges = gp.tuplelist((i, j) for i, j in vals.keys() if vals[i, j] > 0.5)
    unvisited = list(range(n))
    cycle = range(n + 1)  # Cycle of length n+1 so that n is within range
    while unvisited:  # True if the list is not empty
        thiscycle = []
        neighbors = unvisited
        while neighbors:
            current = neighbors[0]
            thiscycle.append(current)
            unvisited.remove(current)
            neighbors = [j for i, j in edges.select(current, '*') if j in unvisited]
        if len(cycle) > len(thiscycle):
            cycle = thiscycle
    return cycle

# Function to compute the distance between two geographic coordinates using the Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth's radius in kilometers
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance = round(R * c)
    return distance

# Compute distances between cities using the Haversine formula
dist = {
    (i, j): haversine(cities[i][1], cities[i][2], cities[j][1], cities[j][2])
    for i in range(n) for j in range(i)
}

In [4]:
m = gp.Model()

vars = m.addVars(dist.keys(), obj=dist, vtype=GRB.BINARY, name='e')

keys_to_add = list(vars.keys())
for i, j in keys_to_add:
    vars[j, i] = vars[i, j]

m.addConstrs(gp.quicksum(vars[i, j] for j in range(n) if j != i) == 2 for i in range(n))

m._vars = vars
m.Params.LazyConstraints = 1
m.write('tsp.lp')
m.optimize(subtourelim)

vals = m.getAttr('X', vars)
tour = subtour(vals)

# Assert final tour is valid
assert len(tour) == n, f"Subtour detected! Only {len(tour)} cities have been covered out of a total of {n}. This solution is not valid for the TSP tour."


print('')
print('Optimal tour: %s' % ' -> '.join([cities[i][0] for i in tour]))
print('Total distance: %g' % m.ObjVal)
print('')


Restricted license - for non-production use only - expires 2027-11-29
Set parameter LazyConstraints to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Non-default parameters:
LazyConstraints  1

Optimize a model with 33 rows, 528 columns and 1056 nonzeros (Min)
Model fingerprint: 0xbfb32f73
Model has 527 linear objective coefficients
Variable types: 0 continuous, 528 integer (528 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [3e+01, 3e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 2e+00]

Found heuristic solution: objective 24342.000000
Presolve time: 0.00s
Presolved: 33 rows, 528 columns, 1056 nonzeros
Variable types: 0 continuous, 528 integer (528 binary)

Root relaxation: objective 7.605500e+03, 52 iterations, 0.00 seconds (0.00 work 

In [5]:
import folium

# Map in Mexico
mexico_map = folium.Map(location=[23.6345, -102.5528], zoom_start=5)

# Marker at each capital city
for city, lat, lon in cities:
    folium.Marker([lat, lon], tooltip=city).add_to(mexico_map)

# Show mapa
mexico_map

In [6]:
# Map in Mexico
mexico_map = folium.Map(location=[23.6345, -102.5528], zoom_start=5)

# Detect tour
optimal_tour_coords = [(cities[i][1], cities[i][2]) for i in tour]
optimal_tour_names = [cities[i][0] for i in tour]

for i in range(n):
    folium.Marker(optimal_tour_coords[i], tooltip=optimal_tour_names[i]).add_to(mexico_map)

# Draw lines
folium.PolyLine(optimal_tour_coords + [optimal_tour_coords[0]], color="blue").add_to(mexico_map)

# Show map
mexico_map